In [ ]:
# OpenBLAS starts one thread per core even for tiny BLAS-1 calls, and the Krylov
# solvers below issue thousands of those per iteration. On a 24-core machine the
# oversubscription cost 4.7x. This has to be set *before* numpy is imported, so if
# numpy is already loaded, restart the kernel.
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import splu, cg, bicgstab, LinearOperator
%matplotlib qt5


In [ ]:
# Inputs
# ------

# Immersed body (circular cylinder) and doubly-periodic domain
R  = 1
Lx = 10
Ly = 10
nx = 81
ny = 81

# Initial timestep (the time derivative acts as the SIMPLEC under-relaxation)
dt = 0.1

# Kinematic viscosity. VAR[IP] is the reduced pressure p/rho, so the density
# never appears explicitly in the equations below.
nu = 0.1

# Constant-flow-rate forcing: the uniform body force in x is adjusted every
# iteration so that the bulk velocity relaxes onto this target. Free slip
# generates no friction drag, so with a *fixed* force the flow would keep
# accelerating and no steady state would exist.
uBulkTarget = 0.5

# Linear solvers. Both give the same answer; only the cost differs.
#   'direct' - SuperLU factorisation of every system, every iteration.
#   'krylov' - BiCGStab on the momentum, preconditioned by a factorisation reused
#              across iterations; CG on the pressure, preconditioned by a
#              constant-coefficient FFT Poisson solve.
# Measured with no-slip (s per iteration, 10 iterations, one thread). Free slip
# needs one matrix per velocity component, so the momentum cost roughly doubles
# and the crossover moves down a little:
#     16k cells   direct 0.15   krylov 0.18
#     66k cells   direct 1.06   krylov 1.08      <- crossover
#    262k cells   direct 12.8   krylov 7.0
# The FFT preconditioner is much cheaper when nx and ny have small prime factors
# (161 = 7*23 costs about twice as much per FFT as 160).
# Free-slip stabilisation (both are on; see doc/freeslip_ibm.pdf):
#   S1  floor the SIMPLEC mobilities rAU, rAtU at V/dt so the pressure laplacian
#       cannot turn indefinite.
#   S3  per-cell adaptive blend of the convective ghost back to the plain
#       treatment, engaged only on cells whose momentum diagonal would otherwise
#       go non-positive -- a handful next to the wall, none elsewhere.
# Together these make ghostConvection = True run at every resolution tested
# (nx = 41...121) on both the coupled and the segregated route.

linearSolver = 'direct'
pTol         = 1e-12       # CG relative tolerance for the pressure equation

# Apply the free-slip ghost to the CONVECTIVE stencil as well as the viscous one.
#
# LEAVE THIS FALSE. It is required for consistency on an oblique wall -- with it,
# uniform flow along a 45-degree free-slip wall is exact (3.7e-09); without it the
# error is O(1) (7.4e-01) -- but every treatment tried so far is unstable on a
# curved body: lagged, lagged with a diagonal repair, with h/delta clipped, and
# fully implicit in the coupled matrix below. Diverges at nx = 41 and 61 in all of
# them, at every timestep tried (10, 1, 0.1), and step rejection does not help.
#
# Cause: the ghost adds  0.5*f*(h/delta)*(n n^T)  to the 2x2 block, where f is the
# face flux. n n^T is rank-1 positive semi-definite, so the *viscous* copy (which
# carries +nu instead of f) is well posed and is exactly what makes delta -> 0 safe
# in Luchini et al. But f changes sign, so the convective copy is indefinite, and
# making an indefinite block implicit does not make it well posed.
#
# Likely cure, not yet tried: upwind the convective ghost, so the ghost value only
# enters on inflow faces and f contributes with a definite sign.
ghostConvection = True

In [ ]:
# Grid, fields and sparsity pattern
# ---------------------------------
IU, IV, IP = 0, 1, 2                 # component indices into VAR

dx, dy = Lx/nx, Ly/ny
V      = dx*dy                       # cell volume
nCells = nx*ny

xp = (np.arange(nx)+0.5)*dx          # cell centres
yp = (np.arange(ny)+0.5)*dy
xf = np.arange(nx+1)*dx              # face centres
yf = np.arange(ny+1)*dy
XP, YP = np.meshgrid(xp, yp)
XF, YF = np.meshgrid(xf, yf)

VAR = np.zeros((3, ny, nx))          # u, v, p

# The 5-point stencil is held as five (ny,nx) coefficient arrays in the order
# P, E, W, N, S -- not as an (nCells,nCells) matrix. Its sparsity pattern never
# changes, so the CSR structure and the permutation from stencil order into CSR
# data order are built once, here. Assembling a matrix is then an O(nnz)
# fancy-index rather than an O(nCells^2) scan over a 99.92%-empty dense array.
ID  = np.arange(nCells).reshape(ny, nx)
ROW = np.stack([ID]*5, -1).ravel()
COL = np.stack([ID, np.roll(ID, -1, 1), np.roll(ID, 1, 1),
                    np.roll(ID, -1, 0), np.roll(ID, 1, 0)], -1).ravel()
assert len(set(zip(ROW.tolist(), COL.tolist()))) == ROW.size, \
    'stencil aliases itself: nx and ny must both be at least 3'

_pattern    = csr_matrix((np.arange(1.0, ROW.size+1), (ROW, COL)),
                         shape=(nCells, nCells))
CSR_PERM    = (_pattern.data-1).astype(np.int64)
CSR_INDICES = _pattern.indices
CSR_INDPTR  = _pattern.indptr


# Coupled (u,v) system. The free-slip cross term couples the two components at the
# *same* cell only, so the block matrix
#
#     [ Auu   diag(cU) ] [u]   [bu]
#     [ diag(cV)   Avv ] [v] = [bv]
#
# has just one extra entry per row: 6 nnz instead of 5. Solving it directly removes
# the lag on the cross term altogether, which is what makes the curved-body case
# stable (lagging it diverges at nx = 41 and 61 whatever the timestep).
IDX  = np.arange(nCells)
ROW2 = np.concatenate([ROW, ROW+nCells, IDX, IDX+nCells])
COL2 = np.concatenate([COL, COL+nCells, IDX+nCells, IDX])


def toCSR2(aU, aV, cU, cV):
    """Assemble the coupled 2N x 2N momentum matrix."""
    dat = np.concatenate([aU.reshape(5, -1).T.ravel(), aV.reshape(5, -1).T.ravel(),
                          cU.ravel(), cV.ravel()])
    return csr_matrix((dat, (ROW2, COL2)), shape=(2*nCells, 2*nCells))


def zeroStencil():
    """An empty set of stencil coefficients, indexed [P|E|W|N|S, iy, ix]."""
    return np.zeros((5, ny, nx))


def toCSR(a):
    """Assemble stencil coefficients into a sparse matrix."""
    return csr_matrix((a.reshape(5, -1).T.ravel()[CSR_PERM], CSR_INDICES, CSR_INDPTR),
                      shape=(nCells, nCells))


def offDiagDot(a, phi):
    """(A - diag(aP)) @ phi, as four neighbour shifts. No matrix is formed."""
    return (a[1]*np.roll(phi, -1, 1) + a[2]*np.roll(phi, 1, 1)
          + a[3]*np.roll(phi, -1, 0) + a[4]*np.roll(phi, 1, 0))


def wrapX(f):
    """Cell-array -> face-array in x: append the periodic image of the first face."""
    return np.concatenate([f, f[:, :1]], axis=1)


def wrapY(f):
    return np.concatenate([f, f[:1, :]], axis=0)

## Free-slip immersed boundary — coupled route

Identical to `SIMPLEC.ipynb` except that the free-slip cross term is solved implicitly in a
coupled (u,v) momentum system rather than lagged. Collocated finite volumes on a uniform, doubly periodic Cartesian grid, implicit Euler,
SIMPLEC as in OpenFOAM's `simpleFoam`. Free slip on the body: **no penetration plus zero
tangential gradient**. Full derivation and numerical evidence in `doc/freeslip_ibm.pdf`.

Zero tangential gradient makes the wall velocity the tangential projection of the first
fluid point, $u_{\rm wall}=(I-nn^{T})u_P$ — which is the single constraint $u\cdot n=0$.
Through the linear ghost extrapolation of **Luchini et al., *JCP* 539 (2025) 114245**, Eq. (6):

$$u_{\rm ghost}=a_U u_P+b\,v_P,\qquad a_U=1-\frac{h}{\delta}n_x^2,\qquad b=-\frac{h}{\delta}n_xn_y$$

The identity projector recovers Eq. (6), i.e. no slip, exactly. `ghostArmCoeffs` builds these
once and the **same coefficients drive the viscous stencil, the convective stencil and the
face fluxes** — which is what makes uniform flow parallel to the wall an exact discrete
solution.

**Four ingredients, each fixing a measured O(1) error:**

1. **Normal projection**, not per-arm pinning. Per-arm pinned both components on cells with
   two cut arms; those acted as no-slip patches and the scheme converged to no slip.
2. **Ghost face fluxes**, not staircase masks. Masks block the tangential flux; the ghost
   makes $\nabla\!\cdot\!\phi$ of a uniform tangential field exact (4.4e-10 vs 1.2e-1).
3. **Ghost convection, upwinded.** The paper notes convective corrections are unnecessary
   for second order — true for no slip, where $u_P\to0$ at the wall. Under free slip $u_P$
   is $O(U)$ and reading the blanked zero out of the solid is an O(1) error. Upwinding is
   essential: the ghost adds $\propto f\frac{h}{\delta}nn^{T}$ to the 2×2 block and $f$
   changes sign, so a central ghost is indefinite. Upwinding picks it up only on inflow
   faces, restoring positive semi-definiteness.
4. **S1 + S3 stabilisation** (see the inputs cell).

### Verification

| 45° free-slip channel (exact: 0) | ghost on | ghost off |
|---|---|---|
| $\max u-\min u$, nx=41 | 2.2e-06 | 1.2e-01 |
| $\max u-\min u$, nx=81 | 3.4e-06 | 1.0e-01 |
| spurious $g_x$, nx=81 | −6.3e-08 | 2.0e-03 |

Ghost-off is **not** merely less accurate: its error is h-independent (observed order 0.1–0.3),
so it converges cleanly to the wrong answer.

Cylinder drag at $Re_R=1.25$, upwind ghost + S1 + S3, all converged to 1e-9:

| nx | 41 | 61 | 81 | 121 |
|---|---|---|---|---|
| segregated | 1.263816 | 1.254353 | 1.250983 | 1.254389 |
| coupled | 1.236070 | 1.249283 | 1.247104 | 1.250610 |

### Still open

Continuity is not machine-zero: removing the face masks makes the body transparent to
tangential flow while the interior stays blanked, leaving a divergence source in the first
solid layer that the projection cannot remove (fluid 4e-06, first solid layer 4.8e-03). The
principled cure is partial face areas $\theta\in[0,1]$, which keeps the body opaque and never
needs a value inside the solid. The symmetric state is also linearly unstable above
$Re_R\approx5$ with this discretisation, so use $\nu\gtrsim0.4$ for convergence studies.

### Why couple

The cross term couples the two components at the **same cell only**, so the block matrix
has just 6 nonzeros per row instead of 5. Coupling removes the lag *and* the artificial
`|cross|` diagonal repair the segregated route needs — and that repair is not free: it
inflates $a_P$, which changes $r_{A_U}/r_{A_tU}$ and hence the **Rhie–Chow flux**. Remove it
and the two routes agree to 4.0e-06; keep it and they differ by 4.6e-03. The coupled route
keeps $a_P$ true.

In [ ]:
# Finite-volume operators
# -----------------------
# `fvm_*` add to the implicit stencil coefficients, `fvc_*` return explicit fields.
# Everything is doubly periodic, so every neighbour access is an np.roll.
#
# The optional `openX`, `openY` face masks close the faces that touch the body.
# A closed face carries no flux of any kind, which is how no-penetration is
# imposed; the same pair of masks must reach fvm_lap, fvc_grdFlux and fvc_grd, or
# pEqn.flux() stops matching the assembled pressure matrix and continuity breaks.
# `GammaY` defaults to `GammaX`: it is only separate because free slip makes the
# momentum diagonal, and hence rAtU, different for u and for v.

def fvm_ddt_LHS(a, dt):
    """Implicit Euler d/dt: adds V/dt to the diagonal."""
    a[0] += V/dt


def fvm_ddt_RHS(rhs, phi, dt):
    """Explicit part of the implicit-Euler d/dt: V*phi^n/dt."""
    rhs += V*phi/dt


def fvm_lap(a, GammaX, GammaY=None, openX=None, openY=None):
    """
    Implicit laplacian(Gamma, .) with linear face interpolation of Gamma.
    Sign convention: negative diagonal, positive off-diagonals.
    """
    if GammaY is None:
        GammaY = GammaX
    gE = 0.5*(GammaX+np.roll(GammaX, -1, 1))*dy/dx
    gW = 0.5*(GammaX+np.roll(GammaX,  1, 1))*dy/dx
    gN = 0.5*(GammaY+np.roll(GammaY, -1, 0))*dx/dy
    gS = 0.5*(GammaY+np.roll(GammaY,  1, 0))*dx/dy
    if openX is not None:
        gE = gE*openX[:, 1:]
        gW = gW*openX[:, :-1]
        gN = gN*openY[1:, :]
        gS = gS*openY[:-1, :]
    a[0] -= gE+gW+gN+gS
    a[1] += gE
    a[2] += gW
    a[3] += gN
    a[4] += gS


def fvm_divConvFlux(a, phiX, phiY):
    """Implicit div(phi u) with central differencing."""
    fE, fW = phiX[:, 1:], phiX[:, :-1]
    fN, fS = phiY[1:, :], phiY[:-1, :]
    a[0] += 0.5*(fE-fW+fN-fS)
    a[1] += 0.5*fE
    a[2] -= 0.5*fW
    a[3] += 0.5*fN
    a[4] -= 0.5*fS


def fvc_grd(phi, openX=None, openY=None):
    """
    Volume-integrated Gauss gradient, sum_f phi_f Sf (that is, V*grad(phi)).
    On a closed face the face value is the owner value, i.e. zero normal gradient,
    so the gradient never reaches into the solid.
    """
    fE = 0.5*(np.roll(phi, -1, 1)+phi)
    fW = 0.5*(phi+np.roll(phi, 1, 1))
    fN = 0.5*(np.roll(phi, -1, 0)+phi)
    fS = 0.5*(phi+np.roll(phi, 1, 0))
    if openX is not None:
        fE = np.where(openX[:, 1:],  fE, phi)
        fW = np.where(openX[:, :-1], fW, phi)
        fN = np.where(openY[1:, :],  fN, phi)
        fS = np.where(openY[:-1, :], fS, phi)
    return np.stack([dy*(fE-fW), dx*(fN-fS)])


def fvc_grdFlux(phi, GammaX, GammaY=None, openX=None, openY=None):
    """Face fluxes Gamma_f * snGrad(phi) * |Sf|, i.e. the flux of fvm_lap(Gamma, phi)."""
    if GammaY is None:
        GammaY = GammaX
    fx = (phi-np.roll(phi, 1, 1))/dx * 0.5*(GammaX+np.roll(GammaX, 1, 1))*dy
    fy = (phi-np.roll(phi, 1, 0))/dy * 0.5*(GammaY+np.roll(GammaY, 1, 0))*dx
    fx, fy = wrapX(fx), wrapY(fy)
    if openX is not None:
        fx, fy = fx*openX, fy*openY
    return fx, fy


def fvc_divFlux(phiX, phiY):
    """Volume-integrated divergence of a face-flux field."""
    return (phiX[:, 1:]-phiX[:, :-1]) + (phiY[1:, :]-phiY[:-1, :])

In [ ]:
# Immersed boundary: free slip
# ----------------------------
# No penetration + zero tangential gradient, to first order.
IBM_SOLID_WEIGHT = 1e12    # penalty enforcing u = 0 in cells inside the body
BISECT_TOL       = 1e-12   # *relative* tolerance for locating the wall on an arm


def levelSet(x, y):
    """Negative inside the body, zero on the surface. Its gradient gives the normal."""
    return (x-Lx/2)**2 + (y-Ly/2)**2 - R**2


def isInBody(x, y):
    return levelSet(x, y) <= 0.0


def surfaceNormal(x, y):
    """
    Unit surface normal from the level-set gradient. Only nX**2, nY**2 and nX*nY
    are ever used, so the sign does not matter. Evaluated at the wall intersection
    of each stencil arm, not at the cell centre: the ghost correction carries a
    1/delta factor, so an O(h) error in the normal would be amplified without
    bound on cells that sit very close to the surface.
    """
    e  = 1e-6*min(dx, dy)
    gX = (levelSet(x+e, y)-levelSet(x-e, y))/(2*e)
    gY = (levelSet(x, y+e)-levelSet(x, y-e))/(2*e)
    mag = np.sqrt(gX**2+gY**2)
    mag = np.where(mag > 0.0, mag, 1.0)
    return gX/mag, gY/mag


solid = isInBody(XP, YP)
fluid = ~solid

# Face fluxes use the same free-slip ghost extrapolation as the viscous stencil,
# exactly as Luchini et al. do for no slip: no face is masked and no correction is
# made to continuity or pressure. On a face between a fluid and a solid cell the
# solid value is replaced by the ghost, which for the *tangential projection* is
#
#     u_ghost = u_P [1 - (h/delta) nX^2] - (h/delta) nX nY v_P .
#
# For a uniform flow parallel to the wall the projection is the identity, so
# u_ghost = u_P: every one of a cut cell's four full faces then carries the uniform
# flux, and a constant field on a full Cartesian cell has identically zero discrete
# divergence. That is what makes uniform tangential flow exact on an oblique wall,
# which closing the faces on the staircase destroys.
def ghostFaceCoeffs():
    """Per-face coefficients (pU, pV) for the lo and hi side of every face."""
    def coeffs(axis, h, wrap):
        sh   = (1, axis)
        loS, hiS = wrap(np.roll(solid, *sh)), wrap(solid)
        Xlo, Ylo = wrap(np.roll(XP, *sh)), wrap(np.roll(YP, *sh))
        Xhi, Yhi = wrap(XP), wrap(YP)
        sx, sy = (1, 0) if axis == 1 else (0, 1)
        pULo = np.ones_like(Xlo); pVLo = np.zeros_like(Xlo)
        pUHi = np.ones_like(Xlo); pVHi = np.zeros_like(Xlo)
        # hi cell in the solid: extrapolate from lo, forwards
        d  = wallDistance(sx, sy, Xlo, Ylo, h)
        nXf, nYf = surfaceNormal(Xlo+sx*d, Ylo+sy*d)
        r  = h/np.where(d > 0.0, d, 1.0)
        nSelf = nXf if axis == 1 else nYf
        m = loS ^ hiS
        mm = (~loS) & hiS
        pULo = np.where(mm, 1.0+(1.0-r*nSelf**2), pULo)
        pVLo = np.where(mm, -r*nXf*nYf, pVLo)
        pUHi = np.where(mm, 0.0, pUHi); pVHi = np.where(mm, 0.0, pVHi)
        # lo cell in the solid: extrapolate from hi, backwards
        d  = wallDistance(-sx, -sy, Xhi, Yhi, h)
        nXb, nYb = surfaceNormal(Xhi-sx*d, Yhi-sy*d)
        r  = h/np.where(d > 0.0, d, 1.0)
        nSelf = nXb if axis == 1 else nYb
        mm = loS & (~hiS)
        pUHi = np.where(mm, 1.0+(1.0-r*nSelf**2), pUHi)
        pVHi = np.where(mm, -r*nXb*nYb, pVHi)
        pULo = np.where(mm, 0.0, pULo); pVLo = np.where(mm, 0.0, pVLo)
        return pULo, pVLo, pUHi, pVHi
    return coeffs(1, dx, wrapX), coeffs(0, dy, wrapY)


def wallDistance(sx, sy, x, y, h):
    """
    Distance in (0,h) from (x,y) to the body surface along the direction (sx,sy),
    by bisection on every point at once. The tolerance is relative to h, so
    refining the mesh refines the wall position with it.
    """
    inAt0 = isInBody(x, y)
    sIn   = np.where(inAt0, 0.0, h)
    sOut  = np.where(inAt0, h, 0.0)
    for _ in range(int(np.ceil(-np.log2(BISECT_TOL)))+1):
        sMid = 0.5*(sIn+sOut)
        hit  = isInBody(x+sx*sMid, y+sy*sMid)
        sIn  = np.where(hit, sMid, sIn)
        sOut = np.where(hit, sOut, sMid)
    return 0.5*(sIn+sOut)


ARMS = [(1, -1, 1, dx,  1,  0),      # East
        (2,  1, 1, dx, -1,  0),      # West
        (3, -1, 0, dy,  0,  1),      # North
        (4,  1, 0, dy,  0, -1)]      # South


def ghostArmCoeffs(rMax=np.inf):
    """
    Per-cell, per-arm coefficients of the free-slip ghost value, optionally with
    h/delta clipped at rMax (see ARM_CONV below):

        u_ghost = aU*u_P + b*v_P ,      aU = 1 - (h/delta) nX^2
        v_ghost = aV*v_P + b*u_P ,      aV = 1 - (h/delta) nY^2 ,  b = -(h/delta) nX nY

    i.e. the tangential projection u_wall = (I - n n^T) u_P fed through the linear
    ghost extrapolation of Luchini et al. (2025) Eq. (6). The identity projector
    (aU = aV = 1 - h/delta, b = 0) recovers Eq. (6), i.e. no slip.

    The same coefficients are used by the viscous stencil, the convective stencil
    and the face fluxes. That is what makes a uniform flow parallel to the wall an
    exact discrete solution: for such a field u_wall = u_P, so every ghost equals
    its own cell value and every operator sees a constant field.
    """
    out = []
    for ax, shift, axis, h, sx, sy in ARMS:
        cut = fluid & np.roll(solid, shift, axis)
        delta  = wallDistance(sx, sy, XP, YP, h)
        nX, nY = surfaceNormal(XP+sx*delta, YP+sy*delta)
        r = np.where(cut, np.minimum(h/delta, rMax), 0.0)
        out.append((cut, 1.0-r*nX**2, 1.0-r*nY**2, -r*nX*nY))
    return out


def ibm_lapFreeSlip(Gamma):
    """Viscous stencils for u and v, plus the cross coefficient (see ghostArmCoeffs)."""
    base = zeroStencil()
    fvm_lap(base, Gamma)
    laplU, laplV = base.copy(), base.copy()
    cross = np.zeros((ny, nx))
    for (ax, _, _, _, _, _), (cut, aU, aV, b) in zip(ARMS, ARM):
        if not cut.any():
            continue
        d1 = base[ax]
        laplU[0] = np.where(cut, laplU[0]+d1*aU, laplU[0])
        laplV[0] = np.where(cut, laplV[0]+d1*aV, laplV[0])
        cross    = np.where(cut, cross+d1*b, cross)
        laplU[ax] = np.where(cut, 0.0, laplU[ax])
        laplV[ax] = np.where(cut, 0.0, laplV[ax])
    return laplU, laplV, cross


def fvm_divConvFluxIBM(aU_, aV_, phiX, phiY):
    """
    Implicit div(phi u) with the free-slip ghost on the cut arms, upwinded, and
    blended cell-by-cell back to the plain treatment only where it would otherwise
    destroy positivity of the momentum diagonal.

    On a cut arm the upwind inflow term has no off-diagonal to land on -- the
    neighbour is solid -- so it falls on the diagonal and can push it non-positive
    (2 cells at nx = 41), which makes rAtU negative and the pressure laplacian
    indefinite. Per cut cell we therefore take

        contribution = w*(ghost upwind) + (1-w)*(plain central),

    with w in [0,1] the largest value that keeps the diagonal at or above a floor.
    The diagonal is linear in w, so w follows in closed form. w = 1 -- full ghost,
    fully consistent -- on every cell that does not need help, which is all of them
    bar a handful next to the wall. That is the advantage over a global Peclet
    blend, which gives up consistency everywhere to rescue a few cells.
    """
    nC = len(ARMS)
    dG = np.zeros((2, ny, nx))      # diagonal, ghost-upwind form
    dP = np.zeros((2, ny, nx))      # diagonal, plain central form
    oP = np.zeros((nC, ny, nx))     # off-diagonal, plain form only
    cr = np.zeros((ny, nx))         # cross, ghost form only
    dBase = np.zeros((2, ny, nx))   # non-cut arms, common to both
    oBase = np.zeros((nC, ny, nx))
    for k in range(nC):
        ax = ARMS[k][0]
        cut, ag, agV, b = ARM_CONV[k]
        f = {1: phiX[:, 1:], 2: -phiX[:, :-1], 3: phiY[1:, :], 4: -phiY[:-1, :]}[ax]
        nc = ~cut
        dBase[0] += np.where(nc, 0.5*f, 0.0)
        dBase[1] += np.where(nc, 0.5*f, 0.0)
        oBase[k] = np.where(nc, 0.5*f, 0.0)
        fOut = np.where(cut, np.maximum(f, 0.0), 0.0)
        fIn = np.where(cut, np.minimum(f, 0.0), 0.0)
        dG[0] += fOut + fIn*ag
        dG[1] += fOut + fIn*agV
        cr -= fIn*b
        dP[0] += np.where(cut, 0.5*f, 0.0)
        dP[1] += np.where(cut, 0.5*f, 0.0)
        oP[k] = np.where(cut, 0.5*f, 0.0)
    w = np.ones((ny, nx))
    for c in range(2):
        aDiag = (aU_ if c == 0 else aV_)[0]
        floor = 0.25*(aDiag + dBase[c])
        d0 = aDiag + dBase[c] + dP[c]
        slope = dG[c] - dP[c]
        need = floor - d0
        bad = (slope < 0.0) & (need > slope)
        safe = np.where(slope != 0.0, slope, 1.0)
        w = np.where(bad, np.minimum(w, np.clip(need/safe, 0.0, 1.0)), w)
    aU_[0] += dBase[0] + w*dG[0] + (1.0 - w)*dP[0]
    aV_[0] += dBase[1] + w*dG[1] + (1.0 - w)*dP[1]
    for k in range(nC):
        ax = ARMS[k][0]
        aU_[ax] += oBase[k] + (1.0 - w)*oP[k]
        aV_[ax] += oBase[k] + (1.0 - w)*oP[k]
    return w*cr, w*cr


def ibm_blankSolid(a):
    """
    Enforce u = 0 in cells whose centre lies inside the body.

    Applied to the *assembled* momentum stencil, so that the blanked diagonal
    keeps the same (positive) sign as every fluid diagonal. Folding it into the
    laplacian instead would leave -IBM_SOLID_WEIGHT on the diagonal, because the
    laplacian is subtracted, and hence a negative rAU / rAtU inside the solid.
    With every face closed, these cells are decoupled from the fluid anyway and
    the value is arbitrary; zero keeps the diagnostics readable.
    """
    a[0] += np.where(solid, IBM_SOLID_WEIGHT, 0.0)


def interpolatePhi(Ux, Uy):
    """Face volume fluxes, with the free-slip ghost used on faces touching the body."""
    (aU, aV, bU, bV), (cU, cV, dU, dV) = FACE
    phiX = 0.5*(aU*wrapX(np.roll(Ux, 1, 1)) + aV*wrapX(np.roll(Uy, 1, 1))
              + bU*wrapX(Ux)               + bV*wrapX(Uy))*dy
    phiY = 0.5*(cU*wrapY(np.roll(Uy, 1, 0)) + cV*wrapY(np.roll(Ux, 1, 0))
              + dU*wrapY(Uy)               + dV*wrapY(Ux))*dx
    return phiX, phiY


ARM = ghostArmCoeffs()

# The convective ghost needs delta clipped, the viscous one must not be.
#
# On a curved body delta can be arbitrarily small. The viscous ghost enters as
# laplU[0] += d1*aU and the laplacian is *subtracted*, so a large negative
# aU = 1 - (h/delta) nX^2 makes the momentum diagonal large and positive -- this is
# exactly the property Luchini et al. rely on, and it must be left alone. The
# convective ghost enters as 0.5*f*aU with f the face flux, of either sign, so the
# same large negative aU can flip the diagonal instead. Measured on the cylinder:
#
#     nx    min delta/h   max h/delta   aU min    min diag   #(diag<=0)
#     41      0.0238         42.0       -38.5      -29.8         2      -> diverges
#     81      0.1508          6.6        -5.2      +0.165        0      -> converges
#
# Clipping only the convective copy keeps the viscous limit exact. It is consistent
# with the method's own premise (Sec. 3.1: viscous terms dominate near the wall, so
# the convective correction is the small one), and it perturbs the wall position
# only for cells closer than rConvMax to it, an O(h) effect at first order.
rConvMax = 5.0
ARM_CONV = ghostArmCoeffs(rMax=rConvMax)
FACE = ghostFaceCoeffs()

# Fluid cells touching the body, used for the slip diagnostic below
nearBody = fluid & (np.roll(solid, 1, 1) | np.roll(solid, -1, 1) |
                    np.roll(solid, 1, 0) | np.roll(solid, -1, 0))

In [ ]:
# Linear solvers
# --------------
# Both classes below only ever *precondition*. The matrices that are actually
# solved -- and therefore pEqn.flux() and the whole SIMPLEC splitting -- are
# untouched, so the algorithm stays exactly the OpenFOAM one.

class ReusedLU:
    """
    A SuperLU factorisation kept as a preconditioner across SIMPLE iterations.

    The momentum matrix changes a little every iteration (through the convective
    flux and dt), so the factorisation goes stale -- but a stale LU is still an
    excellent preconditioner, and the Krylov solve remains exact. Refactorise only
    when the iteration count says the old one has decayed. Because the LU is never
    used as the solver, the answer cannot depend on how stale it is.
    """

    def __init__(self, krylov, maxIts=20, rtol=1e-12):
        self.krylov, self.maxIts, self.rtol = krylov, maxIts, rtol
        self.lu, self.nFactor, self.nSolve, self.nIts = None, 0, 0, 0

    def solve(self, A, b, x0=None):
        if self.lu is None:
            self.lu = splu(A.tocsc()); self.nFactor += 1
        for attempt in range(2):
            its = [0]
            x, info = self.krylov(
                A, b, x0=x0, rtol=self.rtol, maxiter=500,
                M=LinearOperator(A.shape, matvec=self.lu.solve),
                callback=lambda xk: its.__setitem__(0, its[0]+1))
            if info == 0 and its[0] <= self.maxIts:
                break
            if attempt == 0:                                   # stale: refresh, retry
                self.lu = splu(A.tocsc()); self.nFactor += 1
        assert info == 0, f'momentum {self.krylov.__name__} failed, info={info}'
        self.nSolve += 1
        self.nIts += its[0]
        return x


class FftPoissonPC:
    """
    Preconditioner for the pressure equation: the *constant*-coefficient periodic
    Poisson operator with the same 5-point stencil, inverted exactly by FFT in
    O(N log N), wrapped in a symmetric Jacobi scaling.

    The Jacobi scaling is what makes this work at all: without it the pressure
    matrix has condition number ~1e17, because rAtU is ~1e-12 inside the solid
    (the IBM penalty) against ~1e2 in the fluid. That is a pure row/column scaling
    artefact -- symmetric Jacobi scaling alone brings it to ~5e4 -- and the FFT
    then removes the smooth error that Jacobi leaves behind.
    """

    def __init__(self):
        tx = 2*np.pi*np.fft.fftfreq(nx)
        ty = 2*np.pi*np.fft.fftfreq(ny)
        TX, TY = np.meshgrid(tx, ty)
        self.symbol = -2*((dy/dx)*(np.cos(TX)-1) + (dx/dy)*(np.cos(TY)-1))
        self.symbol[0, 0] = self.symbol[self.symbol > 0].min()   # keep it SPD

    def update(self, A, GammaBar):
        self.lam = GammaBar*self.symbol
        self.s   = 1.0/np.sqrt(np.abs(A.diagonal()))

    def matvec(self, r):
        z = self.s*r
        z = np.real(np.fft.ifft2(np.fft.fft2(z.reshape(ny, nx))/self.lam)).ravel()
        return self.s*z


def solvePressure(A, b, x0, pc):
    """CG on the SPD system (-A)p = (-b); fvm_lap has a negative diagonal."""
    its = [0]
    p, info = cg(-A, -b, x0=x0, rtol=pTol, maxiter=2000,
                 M=LinearOperator(A.shape, matvec=pc.matvec),
                 callback=lambda xk: its.__setitem__(0, its[0]+1))
    assert info == 0, f'pressure CG failed, info={info}'
    return p, its[0]

In [ ]:
# Run controls
# ------------
startNew    = True
readRestart = False
consistent  = True          # True: SIMPLEC, False: SIMPLE

nIterations = 400
cflMax      = 100.0         # Courant cap on the pseudo-transient relaxation
foMax       = 65.0          # diffusive (Fourier) cap, nu*dt/h^2
dtMax       = 10.0          # hard cap
dtRelax     = 1.0           # backoff factor, driven by the residual (see the loop)
residOld    = np.inf
dtCflPrev   = 1.0

# Initial condition
if readRestart and not startNew:
    try:
        VAR      = np.fromfile('var.fld').reshape(VAR.shape)
        phiHbyAx = np.fromfile('phiHbyAx.fld').reshape((ny, nx+1))
        phiHbyAy = np.fromfile('phiHbyAy.fld').reshape((ny+1, nx))
    except (FileNotFoundError, ValueError):
        print('No restart field found, starting from rest.')
        startNew = True
if startNew:
    VAR[...] = 0.0
    phiHbyAx, phiHbyAy = interpolatePhi(VAR[IU], VAR[IV])

# Static part of the system. Free slip needs one laplacian per velocity component:
# along each stencil arm the staircase normal is that arm's axis, so the component
# along the arm sees no penetration and the component across it sees zero gradient.
laplU, laplV, crossLapl = ibm_lapFreeSlip(nu*np.ones((ny, nx)))
lapl = (laplU, laplV)

# A fluid cell that lost every viscous neighbour would be held only by V/dt, which
# is 0.3% of the diagonal at dtMax. Catch it rather than discover it as divergence.
for iC, l in enumerate(lapl):
    orphan = fluid & (np.abs(l[1:]).sum(axis=0) == 0.0)
    assert not orphan.any(), f'{orphan.sum()} fluid cells have no viscous coupling ' \
                             f'left for component {iC}: the body is under-resolved'

# Pressure reference, pinned in a fluid cell. With every cut face closed the fluid
# region is a pure-Neumann island, so it needs exactly one reference -- and would
# need one per region if the body ever split the fluid in two.
jRef, iRef = divmod(int(np.flatnonzero(fluid.ravel())[0]), nx)

Afluid     = fluid.sum()*V
unitSource = np.stack([np.where(fluid, V, 0.0), np.zeros((ny, nx))])   # source for gx = 1
uSolver    = ReusedLU(bicgstab)              # one coupled (u,v) matrix
pPC        = FftPoissonPC()
t          = 0.0


def bulkVelocity(u):
    return np.sum(u*fluid)*V/Afluid


# Timeloop
for it in range(nIterations):

    VARold  = VAR.copy()
    phiOldX, phiOldY = phiHbyAx.copy(), phiHbyAy.copy()

    # Assemble the operators. None of this depends on the driving force.
    # -----------------------------------------------------------------
    aMom = []
    for iC in range(2):
        a = zeroStencil()
        fvm_ddt_LHS(a, dt)                            # d/dt
        a -= lapl[iC]                                 # -laplacian(nu, u), IBM-corrected

        aMom.append(a)
    if ghostConvection:
        crossConvU, crossConvV = fvm_divConvFluxIBM(aMom[0], aMom[1], phiHbyAx, phiHbyAy)
    else:
        for a in aMom:
            fvm_divConvFlux(a, phiHbyAx, phiHbyAy)
        crossConvU = crossConvV = np.zeros((ny, nx))
    # LHS coefficients of the cross coupling. crossLapl and crossConv were built
    # with the sign for moving to the right-hand side (they used to be lagged), so
    # as implicit matrix entries they take the opposite sign.
    crossU = -(crossLapl + crossConvU)
    crossV = -(crossLapl + crossConvV)
    for a in aMom:
        ibm_blankSolid(a)                             # u = 0 inside the body

    # Split as  aP*u = H(u) - V*grad(p),  H(u) = source - offDiag*u
    aP     = np.stack([a[0] for a in aMom])                # OpenFOAM A(), times V
    rAU    = 1.0/aP
    sumOff = np.stack([a[1:].sum(axis=0) for a in aMom])   # OpenFOAM -H1(), times V
    # S1: keep the SIMPLEC mobilities positive. On a cut arm the upwind inflow
    # term has no off-diagonal to land on, so it falls on the diagonal and can push
    # aP (and hence aP+sumOff) non-positive on a few cells, which makes rAtU
    # negative and turns the pressure laplacian indefinite. Floor both at the pure
    # pseudo-transient value V/dt, which is what an unobstructed cell would have.
    aP     = np.maximum(aP, V/dt)
    rAU    = 1.0/aP
    rAtU   = 1.0/np.maximum(aP+sumOff, V/dt) if consistent else rAU
    rAtUV  = rAtU*V

    # Pressure matrix. rAtU differs between u and v under free slip, so x-faces use
    # the u mobility and y-faces the v one. (OpenFOAM instead averages the
    # components in A(); see addCmptAvBoundaryDiag.)
    pStencil = zeroStencil()
    fvm_lap(pStencil, rAtUV[IU], rAtUV[IV])
    # Every face of a solid cell is closed, so its row came out empty: give it an
    # identity row, with the sign of the negative-definite laplacian.
    pStencil[0][solid] = -1.0
    pStencil[0, jRef, iRef] *= 2.0                    # OpenFOAM fvMatrix::setReference:
    Ap = toCSR(pStencil)                              # symmetric, unlike replacing the row

    # Factorise once; both passes below reuse it.
    Amom = toCSR2(aMom[0], aMom[1], crossU, crossV)
    if linearSolver == 'direct':
        luMom, luP = splu(Amom.tocsc()), splu(Ap.tocsc())
        solveMom = luMom.solve
        solvePres = lambda b: (luP.solve(b), 0)
    else:
        pPC.update(Ap, rAtUV[:, fluid].mean())
        solveMom = lambda b: uSolver.solve(Amom, b)
        solvePres = lambda b: solvePressure(Ap, b, None, pPC)

    def advance(sourceField, pOld):
        """
        One predictor-projection pass with the operators above. The whole pass is
        affine in its source, which is what the flow-rate constraint exploits.
        """
        RHS   = sourceField.reshape(2, -1)
        gradP = fvc_grd(pOld)
        w = solveMom(np.concatenate([RHS[0]-gradP[0].ravel(),
                                     RHS[1]-gradP[1].ravel()]))
        U = np.stack([w[:nCells].reshape(ny, nx), w[nCells:].reshape(ny, nx)])
        # H(u) now also carries the cross coupling, exactly as OpenFOAM's H() does
        HbyA = np.stack([(sourceField[IU] - offDiagDot(aMom[0], U[IU]) - crossU*U[IV])*rAU[IU],
                         (sourceField[IV] - offDiagDot(aMom[1], U[IV]) - crossV*U[IU])*rAU[IV]])
        pX, pY = interpolatePhi(HbyA[IU], HbyA[IV])
        if consistent:
            fX, fY = fvc_grdFlux(pOld, (rAtU[IU]-rAU[IU])*V, (rAtU[IV]-rAU[IV])*V)
            pX, pY = pX+fX, pY+fY
            HbyA = HbyA - (rAU-rAtU)*gradP
        div0 = fvc_divFlux(pX, pY)
        rhsP = div0.copy()
        rhsP[solid] = 0.0
        pNew, its = solvePres(rhsP.ravel())
        pNew = pNew.reshape(ny, nx)
        fX, fY = fvc_grdFlux(pNew, rAtUV[IU], rAtUV[IV])
        return (HbyA - rAtU*fvc_grd(pNew), pNew, pX-fX, pY-fY, div0, its)

    # Constant flow rate, by linear superposition
    # ------------------------------------------
    # Only the source term depends on the driving force gx, and the pass above is
    # affine in it. So one pass with gx = 0 and one with gx = 1 (and everything
    # else zeroed) give the exact gx that lands the bulk velocity on its target --
    # no gain to tune. Free slip leaves the bulk mode almost undamped, so a
    # proportional controller on gx is not a safe alternative here.
    rhs = np.zeros((2, ny, nx))
    for iC in range(2):
        fvm_ddt_RHS(rhs[iC], VAR[iC], dt)
    # Free-slip cross term: the wall value of one component involves the other.
    # Lagged one outer iteration, which keeps the two systems uncoupled and is
    # exact once the outer iteration has converged.
    #
    # Lagging alone is unstable: the cross weight is d(-1)(h/delta) nX nY against a
    # diagonal correction d(-1)(h/delta) nX**2, so wherever |nY| > |nX| on a cut
    # x-arm the explicit term dominates the implicit one and the outer iteration
    # stalls (measured: residual stuck near 2e-4 by nx=121). So |cross| is added to
    # the diagonal and taken off again on the right with the lagged value of the
    # *same* component: the fixed point is untouched, but the iteration matrix
    # regains its diagonal dominance.

    base = advance(rhs,        VAR[IP])
    resp = advance(unitSource, np.zeros((ny, nx)))
    gx   = (uBulkTarget-bulkVelocity(base[0][IU]))/bulkVelocity(resp[0][IU])

    VAR[0:2] = base[0] + gx*resp[0]
    VAR[IP]  = base[1] + gx*resp[1]
    phiHbyAx = base[2] + gx*resp[2]
    phiHbyAy = base[3] + gx*resp[3]
    meanDiv0 = np.mean(np.abs(base[4] + gx*resp[4]))
    pIts     = base[5] + resp[5]

    # Step control: reject and retry rather than merely slowing down
    # ---------------------------------------------------------------
    # A backoff that only shrinks the *next* dt cannot help once a step has
    # already destroyed the state, and with free slip the momentum operator is
    # near-singular at large dt, so a bad step can grow the solution by 100x at
    # once. Roll the step back instead and repeat it smaller.
    resid = np.mean(np.abs(VARold[0:2]-VAR[0:2]))
    if not np.isfinite(resid) or resid > 10.0*residOld:
        VAR[...] = VARold
        phiHbyAx, phiHbyAy = phiOldX, phiOldY
        dtRelax = max(0.5*dtRelax, 1e-4)
        dt = dtRelax*min(dtCflPrev, dtMax)
        continue

    t += dt

    # Diagnostics
    # -----------
    divU    = fvc_divFlux(phiHbyAx, phiHbyAy)
    invTime = max(np.max(np.abs(VAR[IU]))/dx, np.max(np.abs(VAR[IV]))/dy)

    print(f'Iteration: {it}   t = {t:.3f}')
    print(f'------------------')
    print(f'                     Courant number of this step: {dt*invTime:.2f}')
    print(f'                               timestep used, dt: {dt:.3f}')
    print(f'    global continuity error: {np.sum(divU):.3e}   (sum of div(flux))')
    print(f'     final local continuity error: {np.mean(np.abs(divU)):.3e}')
    print(f'   initial local continuity error: {meanDiv0:.3e}')
    print(f'              bulk velocity / target: {bulkVelocity(VAR[IU]):.9f} / {uBulkTarget}')
    print(f'                     driving force gx: {gx:.6e}')
    print(f'       maximal velocity in field: {np.max(np.sqrt(VAR[IU]**2+VAR[IV]**2)):.6f}')
    print(f'       slip velocity at the body: {np.max(np.abs(VAR[0:2][:, nearBody])):.6f}')
    print(f'           flux through the body: (unmasked: paper design, not enforced)')
    print(f'                 solution change: {np.mean(np.abs(VARold[0:2]-VAR[0:2])):.6e}')
    print(f'              cell Peclet / dtRelax: {invTime*min(dx,dy)**2/nu:.3f} / {dtRelax:.3f}')
    if linearSolver != 'direct':
        # uSolver is a single ReusedLU on the coupled route and a pair on the
        # segregated one; accept either.
        us = uSolver if isinstance(uSolver, list) else [uSolver]
        print(f'      linear solver: {pIts} CG its (p), '
              f'{sum(s.nIts for s in us)/max(sum(s.nSolve for s in us), 1):.1f} '
              f'BiCGStab its/solve (U), '
              f'{sum(s.nFactor for s in us)} factorisations so far')
    print(f'')

    # Pseudo-transient timestep for the next iteration
    # ----------------------------------------------
    # Three caps. The Courant one is the usual advective limit. The cell-Peclet
    # one guards central differencing: above Pe = 2 it loses diagonal dominance,
    # and the lagged free-slip cross term then has no margin left to be absorbed
    # into. The third is a backoff on the residual, which is what actually makes
    # this robust: with an *exact* free-slip condition a uniform tangential flow
    # lies in the null space of the steady momentum operator, so the operator is
    # nearly singular as dt grows, and how far dt can be pushed is not predicted
    # by any fixed formula (measured: the 45-degree channel diverges at dt = 10
    # for nx = 41 but converges for nx = 81, at lower cell Peclet).
    cellPe = invTime*min(dx, dy)**2/nu
    dtPe   = foMax*min(dx, dy)**2/nu/max(1.0, 0.5*cellPe)
    dtCfl  = dtMax if invTime == 0.0 else cflMax/invTime
    dtRelax = max(0.5*dtRelax, 1e-2) if resid > 1.2*residOld else min(1.5*dtRelax, 1.0)
    residOld = resid
    dtCflPrev = min(dtCfl, dtPe)
    dt = dtRelax*min(dtCfl, dtPe, dtMax)

# Save solution
VAR.tofile('var.fld')
phiHbyAx.tofile('phiHbyAx.fld')
phiHbyAy.tofile('phiHbyAy.fld')

In [ ]:
plt.figure()
plt.title('Volumes (green), fluxes (black & red) and pressure (colormap)')
xmin, xmax = Lx/2-1.3*R, Lx/2+1.3*R
ymin, ymax = Ly/2-1.3*R, Ly/2+1.3*R

theta = np.linspace(0, 2*np.pi, 100)
xBody = Lx/2 + R*np.cos(theta)
yBody = Ly/2 + R*np.sin(theta)

plt.plot(xBody, yBody, 'y')
plt.pcolormesh(XP, YP, VAR[IP])
for x in xf:
    plt.plot([x, x], [yf[0], yf[-1]], 'g', lw=0.5)
for y in yf:
    plt.plot([xf[0], xf[-1]], [y, y], 'g', lw=0.5)
plt.quiver(xf, yp, phiHbyAx, np.zeros_like(phiHbyAx), pivot='mid', scale=0.5)
plt.quiver(xp, yf, np.zeros_like(phiHbyAy), phiHbyAy, color='r', pivot='mid', scale=0.5)
for ix, x in enumerate(xp):
    for iy, y in enumerate(yp):
        if xmin < x < xmax and ymin < y < ymax:
            plt.text(x, y, f'{divU[iy, ix]:.1E}', horizontalalignment='center')
plt.colorbar()
plt.xlim([xmin, xmax])
plt.ylim([ymin, ymax])

In [ ]:
plt.figure()
plt.title('Colormap: p, vectors: u')
plt.pcolormesh(XP, YP, VAR[IP])
plt.colorbar()
plt.quiver(XP, YP, VAR[IU], VAR[IV])
plt.plot(xBody, yBody, 'y')

plt.figure()
plt.title('Colormap: u')
plt.pcolormesh(XP, YP, VAR[IU])
plt.colorbar()
plt.plot(xBody, yBody, 'y')

# Slip velocity around the body. On a grid-aligned wall this is exact; on a curved
# one the cells with both arms cut are pinned by two no-penetration conditions and
# slip much less -- see the markdown cell above.
jj, ii = np.where(nearBody)
th = np.arctan2(YP[jj, ii]-Ly/2, XP[jj, ii]-Lx/2)
ut = -np.sin(th)*VAR[IU][jj, ii] + np.cos(th)*VAR[IV][jj, ii]
un = np.cos(th)*VAR[IU][jj, ii] + np.sin(th)*VAR[IV][jj, ii]
corner = ((np.roll(solid, -1, 1) | np.roll(solid, 1, 1)) &
          (np.roll(solid, -1, 0) | np.roll(solid, 1, 0)))[jj, ii]

plt.figure()
plt.title('Velocity in the first fluid cell around the body')
plt.plot(np.degrees(th[~corner]), ut[~corner], 'o', label='tangential, one axis cut')
plt.plot(np.degrees(th[corner]),  ut[corner],  's', label='tangential, both axes cut')
plt.plot(np.degrees(th), un, '.', color='0.6', label='normal')
plt.axhline(0, color='k', lw=0.5)
plt.xlabel('angle around the body [deg]')
plt.ylabel('velocity')
plt.legend()

plt.figure()
plt.title('Colormap: div(phi)')
plt.pcolormesh(XP, YP, divU)
plt.colorbar()